## Complete Example: Decoding Threonine (THR)

Let's work through a complete decoding example. Imagine our model generated a new residue, assigning coordinates to all 14 atoms (backbone and virtual). We start with a 14×3 tensor representing the 3D coordinates of each atom:

**What we expect for THR:**
- THR has **7 real atoms**: N, Cα, C, O, CB, OG1, CG2
- THR has **7 virtual atoms** that encode its identity
- The virtual atoms should be placed: **3 on N** and **4 on O**
- This gives the unique placement code: **(3, 0, 0, 4)**

Let's see if our decoding algorithm correctly identifies this as THR!

### Mapping definitions

In [ ]:
import torch

In [ ]:
# Residue to atoms mapping
ref_atoms = {
    "PAD": [],
    "UNK": ["N", "CA", "C", "O", "CB"],
    "-": [],
    "GLY": ["N", "CA", "C", "O"],  # 0
    "ALA": ["N", "CA", "C", "O", "CB"], # 1
    "CYS": ["N", "CA", "C", "O", "CB", "SG"], # 2
    "SER": ["N", "CA", "C", "O", "CB", "OG"], # 2
    "PRO": ["N", "CA", "C", "O", "CB", "CG", "CD"],# 3
    "THR": ["N", "CA", "C", "O", "CB", "OG1", "CG2"],# 3
    "VAL": ["N", "CA", "C", "O", "CB", "CG1", "CG2"],# 3
    "ASN": ["N", "CA", "C", "O", "CB", "CG", "OD1", "ND2"],# 4
    "ASP": ["N", "CA", "C", "O", "CB", "CG", "OD1", "OD2"],# 4
    "ILE": ["N", "CA", "C", "O", "CB", "CG1", "CG2", "CD1"],# 4
    "LEU": ["N", "CA", "C", "O", "CB", "CG", "CD1", "CD2"],# 4
    "MET": ["N", "CA", "C", "O", "CB", "CG", "SD", "CE"],# 4
    "GLN": ["N", "CA", "C", "O", "CB", "CG", "CD", "OE1", "NE2"],# 5
    "GLU": ["N", "CA", "C", "O", "CB", "CG", "CD", "OE1", "OE2"],# 5
    "LYS": ["N", "CA", "C", "O", "CB", "CG", "CD", "CE", "NZ"],# 5
    "HIS": ["N", "CA", "C", "O", "CB", "CG", "ND1", "CD2", "CE1", "NE2"],# 6
    "PHE": ["N", "CA", "C", "O", "CB", "CG", "CD1", "CD2", "CE1", "CE2", "CZ"],# 7
    "ARG": ["N", "CA", "C", "O", "CB", "CG", "CD", "NE", "CZ", "NH1", "NH2"],# 7
    "TYR": ["N", "CA", "C", "O", "CB", "CG", "CD1", "CD2", "CE1", "CE2", "CZ", "OH"], # 8
    "TRP": ["N", "CA", "C", "O", "CB", "CG", "CD1", "CD2", "NE1", "CE2", "CE3", "CZ2", "CZ3", "CH2"],  # 10 noqa: E501
    # ... other definitions to encode input omitted
}

In [ ]:
# Amino acid code based on proximity to backbone atoms (14 atom representation)
fake_atom_placements = {
    "UNK": [".", ".", ".", ".", ".", "N", "N", "N", "N", "N", "N", "N", "N", "N"], # 0
    "GLY": [".", ".", ".", ".", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O"], # 0
    "ALA": [".", ".", ".", ".", ".", "O", "O", "O", "O", "O", "O", "O", "O", "O"], # 1
    "CYS": [".", ".", ".", ".", ".", ".", "O", "O", "O", "O", "O", "O", "O", "O"], # 2
    "SER": [".", ".", ".", ".", ".", ".", "N", "N", "N", "N", "N", "N", "N", "N"], # 2
    "PRO": [".", ".", ".", ".", ".", ".", ".", "O", "O", "O", "O", "O", "O", "O"], # 3
    "THR": [".", ".", ".", ".", ".", ".", ".", "N", "N", "N", "O", "O", "O", "O"], # 3
    "VAL": [".", ".", ".", ".", ".", ".", ".", "N", "N", "N", "N", "N", "N", "N"], # 3
    "ILE": [".", ".", ".", ".", ".", ".", ".", ".", "O", "O", "O", "O", "O", "O"], # 4
    "ASN": [".", ".", ".", ".", ".", ".", ".", ".", "N", "O", "O", "O", "O", "O"], # 4
    "ASP": [".", ".", ".", ".", ".", ".", ".", ".", "N", "N", "O", "O", "O", "O"], # 4
    "LEU": [".", ".", ".", ".", ".", ".", ".", ".", "N", "N", "N", "N", "O", "O"], # 4
    "MET": [".", ".", ".", ".", ".", ".", ".", ".", "N", "N", "N", "N", "N", "N"], # 4
    "GLN": [".", ".", ".", ".", ".", ".", ".", ".", ".", "O", "O", "O", "O", "O"], # 5
    "GLU": [".", ".", ".", ".", ".", ".", ".", ".", ".", "N", "N", "O", "O", "O"], # 5
    "LYS": [".", ".", ".", ".", ".", ".", ".", ".", ".", "N", "N", "N", "N", "N"], # 5
    "HIS": [".", ".", ".", ".", ".", ".", ".", ".", ".", ".", "O", "O", "O", "O"], # 6
    "PHE": [".", ".", ".", ".", ".", ".", ".", ".", ".", ".", ".", "O", "O", "O"], # 7
    "ARG": [".", ".", ".", ".", ".", ".", ".", ".", ".", ".", ".", "N", "N", "N"], # 7
    "TYR": [".", ".", ".", ".", ".", ".", ".", ".", ".", ".", ".", ".", "O", "O"], # 8
    "TRP": [".", ".", ".", ".", ".", ".", ".", ".", ".", ".", ".", ".", ".", "."], # 10
}

In [ ]:
token_to_placement_count = {ttype:[placement.count(atom_name) for atom_name in ref_atoms["GLY"]] for ttype, placement in fake_atom_placements.items()}
placement_count_to_token = {tuple(v):k for k,v in token_to_placement_count.items()}

In [ ]:
token_to_placement_count = {ttype:[placement.count(atom_name) for atom_name in ref_atoms["GLY"]] for ttype, placement in fake_atom_placements.items()}
placement_count_to_token = {tuple(v):k for k,v in token_to_placement_count.items()}

print("Mapping between token and placement count:")
token_to_placement_count

Mapping between token and placement count:


{'UNK': [9, 0, 0, 0],
 'GLY': [0, 0, 0, 10],
 'ALA': [0, 0, 0, 9],
 'CYS': [0, 0, 0, 8],
 'SER': [8, 0, 0, 0],
 'PRO': [0, 0, 0, 7],
 'THR': [3, 0, 0, 4],
 'VAL': [7, 0, 0, 0],
 'ILE': [0, 0, 0, 6],
 'ASN': [1, 0, 0, 5],
 'ASP': [2, 0, 0, 4],
 'LEU': [4, 0, 0, 2],
 'MET': [6, 0, 0, 0],
 'GLN': [0, 0, 0, 5],
 'GLU': [2, 0, 0, 3],
 'LYS': [5, 0, 0, 0],
 'HIS': [0, 0, 0, 4],
 'PHE': [0, 0, 0, 3],
 'ARG': [3, 0, 0, 0],
 'TYR': [0, 0, 0, 2],
 'TRP': [0, 0, 0, 0]}

### From denoised input to decoded residue

In [ ]:
# Denoised 14-atom tensor for THR
coords_thr = torch.tensor([
    [ 1.4,  0.1,  0.0],  # Atom 1 (N)   <-- Code Target 1
    [ 0.0,  0.0,  0.0],  # Atom 2 (Cα)
    [-0.5,  1.4,  0.3],  # Atom 3 (C)
    [-1.7,  1.5,  0.4],  # Atom 4 (O)   <-- Code Target 2
    [-0.2, -0.8,  1.4],  # Atom 5 (CB)
    [ 0.6, -1.9,  1.0],  # Atom 6 (OG1)
    [-1.5, -1.3,  2.1],  # Atom 7 (CG2)

    # --- Virtual Atoms Start Here ---
    [ 1.4,  0.1,  0.0],  # Atom 8  (Placed on Atom 1 - N)
    [ 1.4,  0.1,  0.0],  # Atom 9  (Placed on Atom 1 - N)
    [ 1.4,  0.1,  0.0],  # Atom 10 (Placed on Atom 1 - N)
    [-1.7,  1.5,  0.4],  # Atom 11 (Placed on Atom 4 - O)
    [-1.7,  1.5,  0.4],  # Atom 12 (Placed on Atom 4 - O)
    [-1.7,  1.5,  0.4],  # Atom 13 (Placed on Atom 4 - O)
    [-1.7,  1.5,  0.4],  # Atom 14 (Placed on Atom 4 - O)
])

In [ ]:
print("=" * 80)
print("COMPLETE WALKTHROUGH: DECODING THREONINE (THR)")
print("=" * 80)

# Step 1: Split backbone and sidechain
backbone_coords_thr = coords_thr[:4]  # N, Cα, C, O
sidechain_coords_thr = coords_thr[4:]  # 10 atoms (atoms 5-14)

print("\n--- STEP 1: Understanding the Input ---")
print(f"Input: 14 atoms with 3D coordinates")
print(f"  - Backbone atoms (0-3): N, Cα, C, O")
print(f"  - Potential sidechain atoms (4-13): 10 positions")
print(f"\nBackbone coordinates:")
for i, name in enumerate(["N", "Cα", "C", "O"]):
    print(f"  Atom {i+1:2d} ({name:2s}): {backbone_coords_thr[i].tolist()}")

print(f"\nSidechain coordinates (real + virtual):")
for i in range(10):
    print(f"  Atom {i+5:2d}: {sidechain_coords_thr[i].tolist()}")

COMPLETE WALKTHROUGH: DECODING THREONINE (THR)

--- STEP 1: Understanding the Input ---
Input: 14 atoms with 3D coordinates
  - Backbone atoms (0-3): N, Cα, C, O
  - Potential sidechain atoms (4-13): 10 positions

Backbone coordinates:
  Atom  1 (N ): [1.399999976158142, 0.10000000149011612, 0.0]
  Atom  2 (Cα): [0.0, 0.0, 0.0]
  Atom  3 (C ): [-0.5, 1.399999976158142, 0.30000001192092896]
  Atom  4 (O ): [-1.7000000476837158, 1.5, 0.4000000059604645]

Sidechain coordinates (real + virtual):
  Atom  5: [-0.20000000298023224, -0.800000011920929, 1.399999976158142]
  Atom  6: [0.6000000238418579, -1.899999976158142, 1.0]
  Atom  7: [-1.5, -1.2999999523162842, 2.0999999046325684]
  Atom  8: [1.399999976158142, 0.10000000149011612, 0.0]
  Atom  9: [1.399999976158142, 0.10000000149011612, 0.0]
  Atom 10: [1.399999976158142, 0.10000000149011612, 0.0]
  Atom 11: [-1.7000000476837158, 1.5, 0.4000000059604645]
  Atom 12: [-1.7000000476837158, 1.5, 0.4000000059604645]
  Atom 13: [-1.700000047683

In [ ]:
# Step 2: Compute distances
distances_thr = torch.cdist(backbone_coords_thr.unsqueeze(0), sidechain_coords_thr.unsqueeze(0)).squeeze(0)

print("\n" + "-" * 80)
print("--- STEP 2: Compute Distances from Each Sidechain Atom to Backbone Atoms ---")
print("-" * 80)
print(f"\nDistance matrix shape: {distances_thr.shape}")
print(f"\nDistances (rows=backbone atoms, columns=sidechain atoms):")
print(f"                    N (Atom 1)  Cα (Atom 2)  C (Atom 3)  O (Atom 4)")
for i in range(10):
    print(f"  Sidechain Atom {i+5:2d}:  {distances_thr[0,i]:8.3f}    {distances_thr[1,i]:8.3f}    {distances_thr[2,i]:8.3f}    {distances_thr[3,i]:8.3f}")




--------------------------------------------------------------------------------
--- STEP 2: Compute Distances from Each Sidechain Atom to Backbone Atoms ---
--------------------------------------------------------------------------------

Distance matrix shape: torch.Size([4, 10])

Distances (rows=backbone atoms, columns=sidechain atoms):
                    N (Atom 1)  Cα (Atom 2)  C (Atom 3)  O (Atom 4)
  Sidechain Atom  5:     2.309       1.625       2.478       2.922
  Sidechain Atom  6:     2.375       2.229       3.548       4.148
  Sidechain Atom  7:     3.844       2.890       3.396       3.282
  Sidechain Atom  8:     0.000       1.404       2.322       3.425
  Sidechain Atom  9:     0.000       1.404       2.322       3.425
  Sidechain Atom 10:     0.000       1.404       2.322       3.425
  Sidechain Atom 11:     3.425       2.302       1.208       0.000
  Sidechain Atom 12:     3.425       2.302       1.208       0.000
  Sidechain Atom 13:     3.425       2.302       1.20

In [ ]:
# Step 3: Find closest backbone atom
value_thr, argmin_thr = torch.min(distances_thr, dim=0)

print("\n" + "-" * 80)
print("--- STEP 3: Find Closest Backbone Atom for Each Sidechain Atom ---")
print("-" * 80)
backbone_names = ["N", "Cα", "C", "O"]
for i in range(10):
    closest_bb = argmin_thr[i].item()
    dist = value_thr[i].item()
    print(f"  Sidechain Atom {i+5:2d} → closest to {backbone_names[closest_bb]:2s} (distance: {dist:.3f})")


--------------------------------------------------------------------------------
--- STEP 3: Find Closest Backbone Atom for Each Sidechain Atom ---
--------------------------------------------------------------------------------
  Sidechain Atom  5 → closest to Cα (distance: 1.625)
  Sidechain Atom  6 → closest to Cα (distance: 2.229)
  Sidechain Atom  7 → closest to Cα (distance: 2.890)
  Sidechain Atom  8 → closest to N  (distance: 0.000)
  Sidechain Atom  9 → closest to N  (distance: 0.000)
  Sidechain Atom 10 → closest to N  (distance: 0.000)
  Sidechain Atom 11 → closest to O  (distance: 0.000)
  Sidechain Atom 12 → closest to O  (distance: 0.000)
  Sidechain Atom 13 → closest to O  (distance: 0.000)
  Sidechain Atom 14 → closest to O  (distance: 0.000)


In [ ]:
# Step 4: Identify virtual atoms
threshold = 0.5
virtual_mask_thr = value_thr < threshold

print("\n" + "-" * 80)
print("--- STEP 4: Identify Virtual vs Real Atoms ---")
print("-" * 80)
print(f"\nVirtual atom detection (distance < {threshold} Å to backbone):")
for i in range(10):
    is_virtual = virtual_mask_thr[i].item()
    dist = value_thr[i].item()
    status = "VIRTUAL" if is_virtual else "REAL"
    print(f"  Atom {i+5:2d}: distance={dist:.3f} → {status}")


--------------------------------------------------------------------------------
--- STEP 4: Identify Virtual vs Real Atoms ---
--------------------------------------------------------------------------------

Virtual atom detection (distance < 0.5 Å to backbone):
  Atom  5: distance=1.625 → REAL
  Atom  6: distance=2.229 → REAL
  Atom  7: distance=2.890 → REAL
  Atom  8: distance=0.000 → VIRTUAL
  Atom  9: distance=0.000 → VIRTUAL
  Atom 10: distance=0.000 → VIRTUAL
  Atom 11: distance=0.000 → VIRTUAL
  Atom 12: distance=0.000 → VIRTUAL
  Atom 13: distance=0.000 → VIRTUAL
  Atom 14: distance=0.000 → VIRTUAL


In [ ]:
# Step 5: Count placements
virtual_argmin_thr = argmin_thr[virtual_mask_thr]
arange = torch.arange(4)
counts_thr = (virtual_argmin_thr[:, None] == arange[None, :]).sum(0).long()

print("\n" + "-" * 80)
print("--- STEP 5: Count Virtual Atom Placements on Each Backbone Atom ---")
print("-" * 80)
print(f"\nPlacement counts (ONLY virtual atoms):")
print(f"  N  (Atom 1): {counts_thr[0]} virtual atoms")
print(f"  Cα (Atom 2): {counts_thr[1]} virtual atoms")
print(f"  C  (Atom 3): {counts_thr[2]} virtual atoms")
print(f"  O  (Atom 4): {counts_thr[3]} virtual atoms")

count_tuple_thr = tuple(counts_thr.tolist())
print(f"\nCount tuple: {count_tuple_thr}")



--------------------------------------------------------------------------------
--- STEP 5: Count Virtual Atom Placements on Each Backbone Atom ---
--------------------------------------------------------------------------------

Placement counts (ONLY virtual atoms):
  N  (Atom 1): 3 virtual atoms
  Cα (Atom 2): 0 virtual atoms
  C  (Atom 3): 0 virtual atoms
  O  (Atom 4): 4 virtual atoms

Count tuple: (3, 0, 0, 4)


In [ ]:
# Step 6: Look up residue type
print("\n" + "=" * 80)
print("--- STEP 6: Decode Amino Acid Type from Placement Count ---")
print("=" * 80)

residue_type_thr = placement_count_to_token.get(count_tuple_thr, "UNK")
print(f"\nLookup: placement_count_to_token[{count_tuple_thr}] = '{residue_type_thr}'")

# Step 7: Verify with expected pattern
print("\n" + "-" * 80)
print("--- VERIFICATION: Compare with Known THR Pattern ---")
print("-" * 80)
print(f"\nExpected THR pattern from fake_atom_placements:")
thr_placement = fake_atom_placements["THR"]
print(f"  {thr_placement}")

print(f"\nBreaking down THR encoding:")
print(f"  Positions 0-3 (N, Cα, C, O): {thr_placement[:4]}  → Backbone atoms")
print(f"  Positions 4-6 (CB, OG1, CG2): {thr_placement[4:7]}  → REAL sidechain atoms")
print(f"  Positions 7-13: {thr_placement[7:]}  → VIRTUAL atoms")

n_on_n = thr_placement.count('N')
n_on_ca = thr_placement.count('CA')
n_on_c = thr_placement.count('C')
n_on_o = thr_placement.count('O')

print(f"\nVirtual atom placements for THR:")
print(f"  - {n_on_n} atoms placed on N")
print(f"  - {n_on_ca} atoms placed on CA")
print(f"  - {n_on_c} atoms placed on C")
print(f"  - {n_on_o} atoms placed on O")
print(f"  Expected count: ({n_on_n}, {n_on_ca}, {n_on_c}, {n_on_o})")
print(f"  Decoded count:  {count_tuple_thr}")

if count_tuple_thr == (n_on_n, n_on_ca, n_on_c, n_on_o):
    print(f"\n✓ CORRECT! The decoding worked properly.")
else:
    print(f"\n✗ MISMATCH! The count doesn't match THR's expected pattern.")





--- STEP 6: Decode Amino Acid Type from Placement Count ---

Lookup: placement_count_to_token[(3, 0, 0, 4)] = 'THR'

--------------------------------------------------------------------------------
--- VERIFICATION: Compare with Known THR Pattern ---
--------------------------------------------------------------------------------

Expected THR pattern from fake_atom_placements:
  ['.', '.', '.', '.', '.', '.', '.', 'N', 'N', 'N', 'O', 'O', 'O', 'O']

Breaking down THR encoding:
  Positions 0-3 (N, Cα, C, O): ['.', '.', '.', '.']  → Backbone atoms
  Positions 4-6 (CB, OG1, CG2): ['.', '.', '.']  → REAL sidechain atoms
  Positions 7-13: ['N', 'N', 'N', 'O', 'O', 'O', 'O']  → VIRTUAL atoms

Virtual atom placements for THR:
  - 3 atoms placed on N
  - 0 atoms placed on CA
  - 0 atoms placed on C
  - 4 atoms placed on O
  Expected count: (3, 0, 0, 4)
  Decoded count:  (3, 0, 0, 4)

✓ CORRECT! The decoding worked properly.


In [ ]:
# Step 8: Look up real atoms and assign coordinates
print("\n" + "=" * 80)
print("--- STEP 7: Look Up Real Atoms and Assign Coordinates ---")
print("=" * 80)

real_atom_names_thr = ref_atoms.get(residue_type_thr, ["N", "CA", "C", "O", "CB"])
num_real_atoms_thr = len(real_atom_names_thr)

print(f"\nref_atoms['{residue_type_thr}'] = {real_atom_names_thr}")
print(f"Number of real atoms: {num_real_atoms_thr}")

print(f"\nFinal structure for {residue_type_thr}:")
for i, atom_name in enumerate(real_atom_names_thr):
    coord = coords_thr[i]
    print(f"  {atom_name:4s}: {coord.tolist()}")

print(f"\nDiscarded atoms: {14 - num_real_atoms_thr} virtual atoms (atoms {num_real_atoms_thr + 1}-14)")
print(f"These were only used for encoding the amino acid type and are now thrown away.")




--- STEP 7: Look Up Real Atoms and Assign Coordinates ---

ref_atoms['THR'] = ['N', 'CA', 'C', 'O', 'CB', 'OG1', 'CG2']
Number of real atoms: 7

Final structure for THR:
  N   : [1.399999976158142, 0.10000000149011612, 0.0]
  CA  : [0.0, 0.0, 0.0]
  C   : [-0.5, 1.399999976158142, 0.30000001192092896]
  O   : [-1.7000000476837158, 1.5, 0.4000000059604645]
  CB  : [-0.20000000298023224, -0.800000011920929, 1.399999976158142]
  OG1 : [0.6000000238418579, -1.899999976158142, 1.0]
  CG2 : [-1.5, -1.2999999523162842, 2.0999999046325684]

Discarded atoms: 7 virtual atoms (atoms 8-14)
These were only used for encoding the amino acid type and are now thrown away.


In [ ]:
# Final Summary
print("\n" + "=" * 80)
print("FINAL SUMMARY")
print("=" * 80)
print(f"\n✓ Decoded residue: {residue_type_thr}")
print(f"✓ Real atoms: {num_real_atoms_thr} ({', '.join(real_atom_names_thr)})")
print(f"✓ Virtual atoms used: {14 - num_real_atoms_thr} (discarded after decoding)")
print(f"✓ Placement code: {count_tuple_thr}")
print(f"\nKey insight for THR:")
print(f"  - THR has 3 real sidechain atoms (CB, OG1, CG2)")
print(f"  - The remaining 7 positions are filled with virtual atoms")
print(f"  - 3 virtual atoms placed on N, 4 on O → unique signature (3,0,0,4)")
print(f"  - This signature uniquely identifies the residue as THR!")
print("\n" + "=" * 80)


FINAL SUMMARY

✓ Decoded residue: THR
✓ Real atoms: 7 (N, CA, C, O, CB, OG1, CG2)
✓ Virtual atoms used: 7 (discarded after decoding)
✓ Placement code: (3, 0, 0, 4)

Key insight for THR:
  - THR has 3 real sidechain atoms (CB, OG1, CG2)
  - The remaining 7 positions are filled with virtual atoms
  - 3 virtual atoms placed on N, 4 on O → unique signature (3,0,0,4)
  - This signature uniquely identifies the residue as THR!



## 🎓 Recap

### What Just Happened?

We successfully decoded a **Threonine (THR)** residue from the 14-atom representation using the **placement code** system!

---

### The Complete Process

#### **INPUT: 14 Atoms**
- **Atoms 1-4**: Backbone atoms (N, Cα, C, O) - always real
- **Atoms 5-7**: Real sidechain atoms (CB, OG1, CG2) for THR
- **Atoms 8-14**: Virtual atoms used for encoding

#### **STEP 1-3: Distance Computation**
We computed distances from each of the 10 potential sidechain atoms (positions 4-13) to each of the 4 backbone atoms.

#### **STEP 4: Virtual Atom Detection**
- **Real atoms**: Have unique coordinates (distance > threshold to all backbone atoms)
- **Virtual atoms**: Placed exactly ON backbone atoms (distance ≈ 0)

For THR:
- Atoms 5-7: REAL (CB, OG1, CG2 at different positions)
- Atoms 8-10: VIRTUAL (placed on N)
- Atoms 11-14: VIRTUAL (placed on O)

#### **STEP 5: Placement Counting**
We count only the virtual atoms:
- 3 virtual atoms on **N**
- 0 virtual atoms on **Cα**
- 0 virtual atoms on **C**
- 4 virtual atoms on **O**
- **Count tuple**: `(3, 0, 0, 4)`

#### **STEP 6: Dictionary Lookup**
```python
placement_count_to_token[(3, 0, 0, 4)] → "THR"
```

#### **STEP 7: Real Atom Assignment**
```python
ref_atoms["THR"] → ["N", "CA", "C", "O", "CB", "OG1", "CG2"]
```
We take the first 7 coordinates and assign these atom names.